# Entity ***Mints***
+ Layer **silver**
+ Priority **2**
---
### Imports

In [0]:
%run ../common/medallion_functions

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Execution

In [0]:
bz_mints_df = spark.read.table(f"uniswap.bronze.{entity_name}")

In [0]:
sv_mints_df = spark.sql(f"""
    WITH cte_deduplicate_bz AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY _load_timestamp_bronze DESC
            ) AS rn
        FROM {{df}}
    )
    SELECT
        id AS pk_mint_id,
        pool.id AS pool_id,
        token0.id AS token0_id,
        token1.id AS token1_id,
        transaction.id AS tx_id,
        tickLower AS tick_lower_id,
        tickUpper AS tick_upper_id,
        owner AS owner_address,
        sender AS sender_address,
        origin AS origin_address,
        CAST(amount AS DOUBLE) AS liquidity_amount,
        CAST(amount0 AS DOUBLE) AS token0_amount,
        CAST(amount1 AS DOUBLE) AS token1_amount,
        ROUND(
            CAST(amountUSD AS DOUBLE),
            2
        ) AS usd_amount,
        CAST(FROM_UNIXTIME(CAST(timestamp AS INT)) AS TIMESTAMP) AS timestamp,
        CURRENT_TIMESTAMP() AS _load_timestamp_silver
    FROM cte_deduplicate_bz
    WHERE rn = 1
""", df = bz_mints_df)

### Save & exit

In [0]:
load_result = load_entity(sv_mints_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)